In [ ]:

import json
import spacy
from pathlib import Path

# Load English Named Entity Recognition model
nlp = spacy.load("en_core_web_sm")


datasetname= "V1A"
DATASET_PATH = f"./dataset/{datasetname}/{datasetname}.jsonl"


INPUT_FILE = DATASET_PATH
OUTPUT_FILE = "v2b_dataset_clean.jsonl"
REMOVED_FILE = "v2b_removed_names.jsonl"

# Fields to inspect for personal names
FIELDS_TO_CHECK = [
    "conversation",
    "conversation_summary"
]


def contains_person_name(text):
    if not isinstance(text, str) or not text.strip():
        return False

    doc = nlp(text)

    for ent in doc.ents:
        if ent.label_ == "PERSON":
            return True

    return False


def has_person_name(record):
    input_data = record.get("input", {})

    for field in FIELDS_TO_CHECK:
        text = input_data.get(field, "")

        if contains_person_name(text):
            return True

    return False


def clean_dataset():
    kept = 0
    removed = 0

    with (
        open(INPUT_FILE, "r", encoding="utf-8") as infile,
        open(OUTPUT_FILE, "w", encoding="utf-8") as outfile,
        open(REMOVED_FILE, "w", encoding="utf-8") as removedfile
    ):
        for line_number, line in enumerate(infile, start=1):
            if not line.strip():
                continue

            try:
                record = json.loads(line)

                if has_person_name(record):
                    removedfile.write(
                        json.dumps(record, ensure_ascii=False) + "\n"
                    )
                    removed += 1
                else:
                    outfile.write(
                        json.dumps(record, ensure_ascii=False) + "\n"
                    )
                    kept += 1

            except json.JSONDecodeError:
                print(f"Invalid JSON at line {line_number}")

    print("\n========== DATASET CLEANING ==========")
    print(f"Original records processed: {kept + removed}")
    print(f"Records retained: {kept}")
    print(f"Records removed: {removed}")
    print(f"Clean dataset: {OUTPUT_FILE}")
    print(f"Removed records: {REMOVED_FILE}")


if __name__ == "__main__":
    clean_dataset()
